# Flip-Flops — Internal Gates, Edges, and Evolution in Time

A flip-flop samples its input only at a clock **edge**. This notebook draws the **actual internal gate schematic** (not a black box), lets you **step the schematic through time** to watch the captured state change edge by edge, and pairs every circuit with its clock-aligned waveform.

$$Q_{next} = D\big|_{CLK\uparrow}$$


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False,'font.size':9})

ON, OFF = '#c0392b', '#b0b0b0'
def wcol(b): return ON if b else OFF
def wlw(b):  return 2.4 if b else 1.2

def gate_box(ax, x, y, label, w=0.85, h=0.66):
    ax.add_patch(Rectangle((x,y-h/2),w,h, fc='#eef2f7', ec='#34495e', lw=1.3, zorder=2))
    ax.text(x+w/2,y,label,ha='center',va='center',fontsize=7.5,weight='bold',zorder=3)
    return (x,y+h*0.28),(x,y-h*0.28),(x+w,y)
def wire(ax,p0,p1,bit,elbow=True):
    c,lw=wcol(bit),wlw(bit); (x0,y0),(x1,y1)=p0,p1
    if elbow and abs(y0-y1)>1e-6:
        xm=(x0+x1)/2
        ax.plot([x0,xm],[y0,y0],color=c,lw=lw,zorder=1)
        ax.plot([xm,xm],[y0,y1],color=c,lw=lw,zorder=1)
        ax.plot([xm,x1],[y1,y1],color=c,lw=lw,zorder=1)
    else:
        ax.plot([x0,x1],[y0,y1],color=c,lw=lw,zorder=1)
def pin(ax,x,y,name,bit,side='left'):
    ax.scatter([x],[y],s=34,color=wcol(bit),zorder=4)
    dx=-0.22 if side=='left' else 0.16; ha='right' if side=='left' else 'left'
    ax.text(x+dx,y,f'{name}={bit}',ha=ha,va='center',fontsize=8.5,color=wcol(bit),weight='bold')
def clock(n,period=4):
    t=np.arange(n); return ((t//(period//2))%2).astype(int)
def rising_edges(clk): return [i for i in range(1,len(clk)) if clk[i-1]==0 and clk[i]==1]
def waveform(ax,t,sig,label,color,edges=None):
    ax.step(t,sig,where='post',color=color,lw=2)
    ax.set_ylim(-0.3,1.3); ax.set_yticks([0,1])
    ax.set_ylabel(label,rotation=0,ha='right',va='center'); ax.grid(True,alpha=0.3)
    if edges:
        for e in edges: ax.axvline(e,color='#8e44ad',ls=':',lw=1,alpha=0.6)
print('primitives ready')


primitives ready


## SR Flip-Flop — Internal NAND Gates

An edge-triggered SR flip-flop is built from a clocked input stage feeding a NAND latch. The schematic below shows the four NAND gates and lights each wire by its logic level, so you can trace how $S$, $R$ and $CLK$ combine to drive the latch.

$$Q = \overline{\overline{S\cdot CLK}\cdot \overline{Q}}$$


In [2]:
def sr_settle(s_int, r_int, q0):
    q,qb=q0,1-q0
    for _ in range(8):
        q  = 1-(s_int & qb)
        qb = 1-(r_int & q)
    return q,qb
def draw_sr_ff(S,R,CLK,q0):
    s_int = 1-(S & CLK)   # gated S (active low into latch)
    r_int = 1-(R & CLK)
    q,qb = sr_settle(s_int,r_int,q0)
    fig,ax=plt.subplots(figsize=(8,4)); ax.set_xlim(0,10); ax.set_ylim(0,5); ax.axis('off')
    pin(ax,0.4,3.8,'S',S); pin(ax,0.4,1.2,'R',R); pin(ax,0.4,2.5,'CLK',CLK)
    _,_,g1o=gate_box(ax,2.0,3.6,'NAND')   # S,CLK
    _,_,g2o=gate_box(ax,2.0,1.4,'NAND')   # R,CLK
    wire(ax,(0.4,3.8),(2.0,3.78),S); wire(ax,(0.4,2.5),(2.0,3.42),CLK)
    wire(ax,(0.4,1.2),(2.0,1.22),R); wire(ax,(0.4,2.5),(2.0,1.58),CLK)
    _,_,g3o=gate_box(ax,4.4,3.2,'NAND')   # latch top -> Q
    _,_,g4o=gate_box(ax,4.4,1.6,'NAND')   # latch bot -> Qbar
    wire(ax,g1o,(4.4,3.42),s_int); wire(ax,g2o,(4.4,1.38),r_int)
    # cross couple
    wire(ax,g3o,(6.2,3.2),q,elbow=False); wire(ax,g4o,(6.2,1.6),qb,elbow=False)
    ax.plot([6.2,6.6],[3.2,3.2],color=wcol(q),lw=wlw(q)); ax.plot([6.6,6.6],[3.2,1.05],color=wcol(q),lw=wlw(q))
    ax.plot([6.6,4.0],[1.05,1.05],color=wcol(q),lw=wlw(q)); ax.plot([4.0,4.0],[1.05,1.38],color=wcol(q),lw=wlw(q))
    ax.plot([6.2,6.9],[1.6,1.6],color=wcol(qb),lw=wlw(qb)); ax.plot([6.9,6.9],[1.6,3.75],color=wcol(qb),lw=wlw(qb))
    ax.plot([6.9,4.0],[3.75,3.75],color=wcol(qb),lw=wlw(qb)); ax.plot([4.0,4.0],[3.75,3.42],color=wcol(qb),lw=wlw(qb))
    pin(ax,6.6,3.2,'Q',q,side='right'); pin(ax,6.9,1.6,'Q\u0305',qb,side='right')
    gated = 'CLK=1: inputs pass' if CLK else 'CLK=0: latch holds (inputs blocked)'
    ax.set_title(f'SR flip-flop internals  --  {gated}',fontsize=10,color='#2471a3' if CLK else '#7f8c8d')
    plt.tight_layout(); plt.show()
w_S=widgets.ToggleButtons(options=[0,1],value=1,description='S:')
w_R=widgets.ToggleButtons(options=[0,1],value=0,description='R:')
w_CK=widgets.ToggleButtons(options=[0,1],value=1,description='CLK:')
w_q0=widgets.ToggleButtons(options=[0,1],value=0,description='prev Q:')
display(widgets.VBox([w_S,w_R,w_CK,w_q0]),
        widgets.interactive_output(draw_sr_ff,{'S':w_S,'R':w_R,'CLK':w_CK,'q0':w_q0}))


Output()

## Master-Slave D Flip-Flop — Step the Schematic Through Time

A master-slave D flip-flop chains two D latches on opposite clock phases: the **master** is transparent while $CLK=1$ and the **slave** while $CLK=0$. Move the **time slider**: the schematic redraws at each instant, the clock pointer advances, and you watch the master capture then the slave release — the temporal evolution of the circuit itself, not just an output trace.

$$\text{master tracks }D\text{ when }CLK=1,\quad \text{slave copies master when }CLK=0$$


In [3]:
def ms_simulate(D_seq, CLK_seq):
    m=0; s=0; hist=[]
    for D,C in zip(D_seq,CLK_seq):
        if C==1: m=D            # master transparent
        if C==0: s=m            # slave transparent on low
        hist.append((D,C,m,s))
    return hist

def draw_ms(t_idx):
    n=24
    D_seq=((np.sin(np.arange(n)/2.0)>0).astype(int))
    CLK=clock(n,4)
    hist=ms_simulate(D_seq,CLK)
    D,C,m,s=hist[t_idx]
    mb,sb=1-m,1-s
    fig=plt.figure(figsize=(9,5))
    ax=fig.add_axes([0.02,0.42,0.96,0.55]); ax.set_xlim(0,11); ax.set_ylim(0,5); ax.axis('off')
    # inputs
    pin(ax,0.4,3.5,'D',D); pin(ax,0.4,0.8,'CLK',C)
    # MASTER latch (two NAND/AND-ish boxes) transparent when C=1
    m_active = (C==1)
    ax.add_patch(Rectangle((1.6,1.6),2.4,2.6, fc='#eaf3fb' if m_active else '#f2f2f2',
                 ec='#2471a3' if m_active else '#bbb', lw=2, zorder=1))
    ax.text(2.8,4.0,'MASTER',ha='center',fontsize=8,weight='bold',
            color='#2471a3' if m_active else '#999')
    ax.text(2.8,3.7,'(transparent)' if m_active else '(holding)',ha='center',fontsize=7,
            color='#2471a3' if m_active else '#999')
    _,_,gma=gate_box(ax,2.3,2.9,'D-L')
    pin(ax,3.55,2.9,'m',m,side='right')
    wire(ax,(0.4,3.5),(2.3,3.18),D); wire(ax,(0.4,0.8),(2.3,2.62),C)
    # SLAVE latch transparent when C=0
    s_active = (C==0)
    ax.add_patch(Rectangle((5.2,1.6),2.4,2.6, fc='#eafbef' if s_active else '#f2f2f2',
                 ec='#27ae60' if s_active else '#bbb', lw=2, zorder=1))
    ax.text(6.4,4.0,'SLAVE',ha='center',fontsize=8,weight='bold',
            color='#27ae60' if s_active else '#999')
    ax.text(6.4,3.7,'(transparent)' if s_active else '(holding)',ha='center',fontsize=7,
            color='#27ae60' if s_active else '#999')
    _,_,gsa=gate_box(ax,5.9,2.9,'D-L')
    wire(ax,(3.55,2.9),(5.9,3.18),m)
    # inverted clock to slave
    ax.text(4.55,2.2,'CLK\u0305',fontsize=7,color=wcol(1-C),weight='bold')
    wire(ax,(0.4,0.8),(5.9,2.62),1-C)
    wire(ax,(6.75,2.9),(8.8,2.9),s,elbow=False)
    pin(ax,8.8,2.9,'Q',s,side='right')
    ax.set_title(f'master-slave D-FF at t={t_idx}   D={D} CLK={C}  ->  master m={m}, output Q={s}',fontsize=9.5)
    # waveform strip with time pointer
    axw=fig.add_axes([0.08,0.06,0.86,0.3])
    tt=np.arange(n)
    Dh=[h[0] for h in hist]; Ch=[h[1] for h in hist]; Sh=[h[3] for h in hist]
    axw.step(tt,np.array(Ch)+0.0,where='post',color='#34495e',lw=1.5,label='CLK')
    axw.step(tt,np.array(Dh)+1.4,where='post',color='#2471a3',lw=1.5,label='D')
    axw.step(tt,np.array(Sh)+2.8,where='post',color='#c0392b',lw=1.8,label='Q')
    axw.axvline(t_idx,color='#8e44ad',lw=2)
    axw.set_yticks([0.5,1.9,3.3]); axw.set_yticklabels(['CLK','D','Q'])
    axw.set_xlabel('time tick (slider position marked)'); axw.grid(True,alpha=0.3)
    plt.show()

w_t=widgets.IntSlider(value=0,min=0,max=23,description='time t:',layout=widgets.Layout(width='500px'))
display(w_t, widgets.interactive_output(draw_ms,{'t_idx':w_t}))


IntSlider(value=0, description='time t:', layout=Layout(width='500px'), max=23)

Output()

## D Flip-Flop Output Trace — One Sample per Edge

With the internals understood, the behavioural view: at each rising edge $Q$ takes $D$ and holds it for the whole cycle, ignoring intermediate changes. Dotted lines mark the only instants that matter.


In [ ]:
def d_ff_timeline(D_pat,period):
    n=32; base=[int(c) for c in D_pat.ljust(8,'0')[:8]]
    D=np.array([base[i%len(base)] for i in range(n)])
    CLK=clock(n,period); edges=rising_edges(CLK)
    Q=np.zeros(n,dtype=int); q=0
    for i in range(n):
        if i in edges: q=D[i]
        Q[i]=q
    t=np.arange(n)
    fig,axes=plt.subplots(3,1,figsize=(8.5,3.6),sharex=True)
    waveform(axes[0],t,CLK,'CLK','#34495e',edges)
    waveform(axes[1],t,D,'D','#2471a3',edges)
    waveform(axes[2],t,Q,'Q','#c0392b',edges)
    for e in edges: axes[2].scatter([e],[Q[e]],color='#8e44ad',zorder=5,s=30)
    axes[-1].set_xlabel('time tick (dotted = rising edge)')
    plt.tight_layout(); plt.show()
w_Dp=widgets.Text(value='10110010',description='D cycle:',layout=widgets.Layout(width='420px'))
w_pp=widgets.IntSlider(value=4,min=2,max=8,step=2,description='CLK period:')
display(widgets.VBox([w_Dp,w_pp]),widgets.interactive_output(d_ff_timeline,{'D_pat':w_Dp,'period':w_pp}))


## Latch vs Flip-Flop — The Decisive Plot

Same $D$, same clock/enable. The latch is transparent through the whole high phase; the flip-flop updates once per edge. This is the entire level-vs-edge distinction in one figure.


In [4]:
def latch_vs_ff(period):
    n=32; rng=np.random.default_rng(7)
    D=((np.sin(np.arange(n)/2.5)>0).astype(int)^(rng.random(n)<0.08).astype(int))
    CLK=clock(n,period); edges=rising_edges(CLK); t=np.arange(n)
    QL=np.zeros(n,dtype=int); ql=0; QF=np.zeros(n,dtype=int); qf=0
    for i in range(n):
        if CLK[i]: ql=D[i]
        if i in edges: qf=D[i]
        QL[i]=ql; QF[i]=qf
    fig,axes=plt.subplots(4,1,figsize=(8.5,4.4),sharex=True)
    waveform(axes[0],t,CLK,'CLK/EN','#34495e',edges)
    waveform(axes[1],t,D,'D','#2471a3',edges)
    waveform(axes[2],t,QL,'Q latch','#e67e22',edges)
    waveform(axes[3],t,QF,'Q FF','#c0392b',edges)
    for i in range(n):
        if CLK[i]: axes[2].axvspan(i,i+1,color='#e67e22',alpha=0.07)
    axes[-1].set_xlabel('time tick')
    plt.tight_layout(); plt.show()
w_p2=widgets.IntSlider(value=6,min=2,max=10,step=2,description='period:')
display(w_p2,widgets.interactive_output(latch_vs_ff,{'period':w_p2}))


IntSlider(value=6, description='period:', max=10, min=2, step=2)

Output()

## JK Flip-Flop — Internal Gates and the Toggle Feedback

The JK flip-flop resolves the forbidden state by feeding $Q$ and $\overline{Q}$ back into the input NANDs, so $J=K=1$ toggles. The schematic shows the feedback paths lit; toggling is highlighted.

$$Q_{next} = J\overline{Q} + \overline{K}Q$$


In [5]:
def jk_next(J,K,q): return (J&(1-q))|((1-K)&q)
def draw_jk(J,K,CLK,q0):
    qn = jk_next(J,K,q0) if CLK else q0
    qb=1-qn
    # input nand gating with feedback
    top = 1-(J & CLK & (1-q0))   # J path gated by Qbar
    bot = 1-(K & CLK & q0)       # K path gated by Q
    fig,ax=plt.subplots(figsize=(8.5,4)); ax.set_xlim(0,11); ax.set_ylim(0,5); ax.axis('off')
    pin(ax,0.4,3.9,'J',J); pin(ax,0.4,1.1,'K',K); pin(ax,0.4,2.5,'CLK',CLK)
    _,_,g1o=gate_box(ax,2.2,3.6,'NAND')
    _,_,g2o=gate_box(ax,2.2,1.4,'NAND')
    wire(ax,(0.4,3.9),(2.2,3.78),J); wire(ax,(0.4,2.5),(2.2,3.42),CLK)
    wire(ax,(0.4,1.1),(2.2,1.22),K); wire(ax,(0.4,2.5),(2.2,1.58),CLK)
    _,_,g3o=gate_box(ax,4.6,3.2,'NAND'); _,_,g4o=gate_box(ax,4.6,1.6,'NAND')
    wire(ax,g1o,(4.6,3.42),top); wire(ax,g2o,(4.6,1.38),bot)
    wire(ax,g3o,(6.4,3.2),qn,elbow=False); wire(ax,g4o,(6.4,1.6),qb,elbow=False)
    # cross couple + feedback to inputs
    ax.plot([6.4,6.8],[3.2,3.2],color=wcol(qn),lw=wlw(qn)); ax.plot([6.8,6.8],[3.2,1.05],color=wcol(qn),lw=wlw(qn))
    ax.plot([6.8,4.2],[1.05,1.05],color=wcol(qn),lw=wlw(qn)); ax.plot([4.2,4.2],[1.05,1.38],color=wcol(qn),lw=wlw(qn))
    ax.plot([6.4,7.1],[1.6,1.6],color=wcol(qb),lw=wlw(qb)); ax.plot([7.1,7.1],[1.6,3.75],color=wcol(qb),lw=wlw(qb))
    ax.plot([7.1,4.2],[3.75,3.75],color=wcol(qb),lw=wlw(qb)); ax.plot([4.2,4.2],[3.75,3.42],color=wcol(qb),lw=wlw(qb))
    # feedback to input gates (dashed long paths)
    ax.annotate('',xy=(2.2,3.46),xytext=(7.1,4.4),arrowprops=dict(arrowstyle='->',color=wcol(qb),lw=1,ls='--',connectionstyle='arc3,rad=0.3'))
    ax.annotate('',xy=(2.2,1.54),xytext=(6.8,0.7),arrowprops=dict(arrowstyle='->',color=wcol(qn),lw=1,ls='--',connectionstyle='arc3,rad=-0.3'))
    pin(ax,6.8,3.2,'Q',qn,side='right'); pin(ax,7.1,1.6,'Q\u0305',qb,side='right')
    act={(0,0):'hold',(0,1):'reset',(1,0):'set',(1,1):'TOGGLE'}[(J,K)]
    ax.set_title(f'JK internals  J={J} K={K} -> {act}   Q:{q0}->{qn}'+('' if CLK else '  (CLK=0 hold)'),
                 fontsize=9.5,color='#8e44ad' if act=='TOGGLE' and CLK else 'black')
    plt.tight_layout(); plt.show()
w_J=widgets.ToggleButtons(options=[0,1],value=1,description='J:')
w_K=widgets.ToggleButtons(options=[0,1],value=1,description='K:')
w_JK_ck=widgets.ToggleButtons(options=[0,1],value=1,description='CLK:')
w_jq0=widgets.ToggleButtons(options=[0,1],value=0,description='prev Q:')
display(widgets.VBox([w_J,w_K,w_JK_ck,w_jq0]),
        widgets.interactive_output(draw_jk,{'J':w_J,'K':w_K,'CLK':w_JK_ck,'q0':w_jq0}))


Output()

## T Flip-Flop as a Frequency Divider

Holding $T=1$, the T flip-flop toggles each edge, so $Q$ runs at half the clock. Chaining halves repeatedly — the direct path to a ripple counter.


In [6]:
def t_divider(period,n_stages):
    n=48; CLK=clock(n,period); edges=rising_edges(CLK); t=np.arange(n)
    fig,axes=plt.subplots(n_stages+1,1,figsize=(8.5,1.0*(n_stages+1)+1),sharex=True)
    waveform(axes[0],t,CLK,'CLK','#34495e',edges)
    colors=['#c0392b','#2471a3','#27ae60','#e67e22']; sig_prev=CLK
    for s in range(n_stages):
        Q=np.zeros(n,dtype=int); q=0
        pe=[i for i in range(1,n) if sig_prev[i-1]==0 and sig_prev[i]==1]
        for i in range(n):
            if i in pe: q^=1
            Q[i]=q
        waveform(axes[s+1],t,Q,f'Q{s} (÷{2**(s+1)})',colors[s%4],pe); sig_prev=Q
    axes[-1].set_xlabel('time tick'); plt.tight_layout(); plt.show()
w_p3=widgets.IntSlider(value=4,min=2,max=6,step=2,description='CLK period:')
w_st=widgets.IntSlider(value=3,min=1,max=4,description='stages:')
display(widgets.VBox([w_p3,w_st]),widgets.interactive_output(t_divider,{'period':w_p3,'n_stages':w_st}))


Output()

## Setup and Hold Time — The Sampling Window

$D$ must be stable $t_{su}$ before and $t_h$ after the edge. A transition inside the window is a timing violation with an undefined captured value.

$$t_{su}\le\text{(stable before)},\qquad t_h\le\text{(stable after)}$$


In [7]:
def setup_hold(d_edge_pos,t_su,t_h):
    clk_edge=10.0; x=np.linspace(0,20,1000)
    CLK=(x>=clk_edge).astype(float); D=(x>=d_edge_pos).astype(float)
    lo,hi=clk_edge-t_su,clk_edge+t_h; viol=lo<=d_edge_pos<=hi
    fig,axes=plt.subplots(2,1,figsize=(8,3.2),sharex=True)
    axes[0].plot(x,CLK,color='#34495e',lw=2); axes[0].set_ylabel('CLK',rotation=0,ha='right')
    axes[1].plot(x,D,color='#c0392b' if viol else '#2471a3',lw=2); axes[1].set_ylabel('D',rotation=0,ha='right')
    for ax in axes:
        ax.axvspan(lo,hi,color='#c0392b',alpha=0.15); ax.axvline(clk_edge,color='#8e44ad',ls=':',lw=1.5)
        ax.set_ylim(-0.2,1.2); ax.set_yticks([0,1]); ax.grid(True,alpha=0.3)
    axes[-1].set_xlabel('time (ns)')
    fig.suptitle('TIMING VIOLATION' if viol else 'clean capture',color='#c0392b' if viol else '#2ca02c',fontsize=10)
    plt.tight_layout(); plt.show()
w_de=widgets.FloatSlider(value=4.0,min=0,max=20,step=0.5,description='D edge ns:')
w_su=widgets.FloatSlider(value=2.0,min=0.5,max=4,step=0.5,description='t_su:')
w_h=widgets.FloatSlider(value=1.0,min=0.5,max=4,step=0.5,description='t_h:')
display(widgets.VBox([w_de,w_su,w_h]),widgets.interactive_output(setup_hold,{'d_edge_pos':w_de,'t_su':w_su,'t_h':w_h}))


Output()

## Metastability — Violating the Window

A transition inside the window can leave the output hovering near threshold before resolving randomly. Resolution probability decays as $e^{-t/\tau}$.


In [ ]:
def metastability(tau,n_traj):
    t=np.linspace(0,10,500); rng=np.random.default_rng(1)
    fig,ax=plt.subplots(figsize=(8,3.4))
    ax.axhline(0.5,color='gray',ls='--',lw=1,alpha=0.6); ax.text(0.1,0.52,'metastable threshold',fontsize=8,color='gray')
    for _ in range(n_traj):
        tr=rng.exponential(tau); fin=rng.integers(0,2)
        y=0.5+(fin-0.5)*(1-np.exp(-(np.maximum(t-tr,0))/0.4))
        y[t<tr]=0.5+0.01*np.sin(40*t[t<tr])
        ax.plot(t,y,lw=1.6,color='#c0392b' if fin else '#2471a3',alpha=0.8)
    ax.set_ylim(-0.1,1.1); ax.set_yticks([0,0.5,1]); ax.set_xlabel('time after edge'); ax.set_ylabel('Q'); ax.grid(True,alpha=0.3)
    ax.set_title(f'metastable resolution, tau={tau:.1f}'); plt.tight_layout(); plt.show()
for ta in [1,3,5]: print(f'P(unresolved after {ta}) ~ {np.exp(-ta/1.5):.3f}')
w_tau=widgets.FloatSlider(value=1.5,min=0.3,max=3,step=0.1,description='tau:')
w_nt=widgets.IntSlider(value=6,min=2,max=12,description='trajectories:')
display(widgets.VBox([w_tau,w_nt]),widgets.interactive_output(metastability,{'tau':w_tau,'n_traj':w_nt}))
